# exp107_01 — Contrato de la mezcla GP funcional sobre los datos de la 107

**Experimento, fuera del pipeline canónico.** El ciclo es el del estudio, Python → MATLAB → Python:

| Paso | Archivo | Qué hace |
|---|---|---|
| 1 | `exp107_01_datos.ipynb` | **este notebook**: lee los artefactos de la **107** y escribe el contrato |
| 2 | `mezcla_gp_iteracion.m` | **entrena en MATLAB**: una cadena por job, `mezcla_gp_train.m` |
| 3 | `exp107_04_evaluacion.ipynb` | convergencia, predicción y las diez métricas contra la 107 |

No se simula nada. Se reutilizan la realización, la base B-spline, la FPCA y la partición de
`escenario_107_1_r01_m04`, de modo que las cifras son comparables una a una con las del
PSBPM-FD de la 107 en `M = 4`.

**El modelo**, con una asignación $S_t$ por curva completa:

$$
c_t\mid S_t=h,\ x_t\sim\mathcal N_K\!\big(A_h^\top x_t,\ \sigma_h^2S(\ell_h)+s_h^2I\big),\qquad
\Pr(S_t=h\mid x_t)=\pi_h(x_t),\quad \eta_h=\alpha_h+\omega_h^\top z_t .
$$

- $c_t=L^\top(\theta_t-\bar\theta)$ son los $K$ coeficientes B-spline **blanqueados** con la Gram ($W=LL^\top$).
  Así $\mu+\Phi_o c_t$ **es** la `curva_suavizada`, el objetivo de evaluación (§6.5), y no se paga
  truncamiento FPCA.
- $z_t=(c_{t-1},\dots,c_{t-L})$ son los rezagos $1..L$ de los **$K$ coeficientes de la base completa**
  (`COVARIABLES = "coeficientes"`), la misma información que usa el FAR. Con `"scores"` son los rezagos
  de los $M$ scores FPCA de la 107, la información del PSBPM-FD.
- $S(\ell)$ es la covarianza de los coeficientes de un GP Matérn de varianza 1 proyectado a la base.

## §1 Imports y rutas

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from model_psbp_fd.utils.rutas import experiment_id, construir_paths
from model_psbp_fd.pipelines import (
    cargar_fpca, cargar_representacion, cargar_escenario,
    cargar_config_evaluacion, cargar_hiperparametros, guardar_config_evaluacion,
)
import mezcla_gp_funcional as mgp

### §1.1 `[CONFIG]`

In [2]:
# -- Identificadores: DEBEN coincidir en exp107_01, mezcla_gp_iteracion.m y exp107_04
BASENAME_107 = "escenario_107"      # corrida fuente: sus datos, su particion, su objetivo
BASENAME     = "experimento_107"    # este experimento: sus propios artefactos
ESCENARIO_ID = 1                    # C-1
REPLICA_ID   = 1
M_COV        = 4                    # punto del barrido de la 107 (su FPCA y el PSBPM-FD de referencia)

EID_107       = experiment_id(BASENAME_107, ESCENARIO_ID, REPLICA_ID, M_COV)
EXPERIMENT_ID = experiment_id(BASENAME, ESCENARIO_ID, REPLICA_ID, M_COV)

N_CHAINS = 2
COVARIABLES = "coeficientes"   # rezagos de c_t (base completa); "scores" = rezagos de xi_1..xi_M

CFG_MEZCLA = mgp.ConfigMezclaGP(
    N=30, nsim=3000, burn=1000, thin=4,
    covarianza="iw_gp_comun", # C comun ~ IW(nu0, (nu0-K-1) GP Matern); "gp" = C_h fija al GP, "iw" = C_h libre
    nucleo="matern32",
    prior_A="encogido",       # A_h ~ MN(B_MCO, diag(v_int, v_pend..), C); "g" colapsa con "iw"
    v_intercepto=10.0, v_pendiente=0.01,
    nu0_extra=3.0,            # nu0 = 17: prior debil, el GP es solo el centro
    lam0="pooled",            # solo "iw"
    prior_mu_alpha="disperso", conc_a=1.0, conc_b=20.0,   # DPM disperso: a ~ G(1, 20), E[a] = 0.05
    n_mov=5,                  # movimientos de etiqueta por tipo e iteracion
)

P107  = construir_paths(EID_107, crear=False)
PATHS = construir_paths(EXPERIMENT_ID, limpiar=True)
print(f"fuente : {EID_107}\ndestino: {EXPERIMENT_ID}")
for k, v in PATHS.items():
    print(f"  {k:<13} {v}")

fuente : escenario_107_1_r01_m04
destino: experimento_107_1_r01_m04
  raw           C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\data\simulaciones\raw\experimento_107_1_r01_m04
  functional    C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\data\simulaciones\processed\functional\experimento_107_1_r01_m04
  predict       C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\data\simulaciones\processed\predict\experimento_107_1_r01_m04
  out_report    C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\reports\simulaciones\experimento_107_1_r01_m04
  out_artefact  C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\artefact\simulaciones\experimento_107_1_r01_m04


## §2 Artefactos de la 107

`n_lags`, `T0`, el objetivo y la semilla base se **leen**; no se redeclaran.

In [3]:
fpca = cargar_fpca(P107)
fr, THETA, fr_cfg = cargar_representacion(P107)
EVAL_107 = cargar_config_evaluacion(P107)
HP_107   = cargar_hiperparametros(P107)
ESC      = cargar_escenario(str(P107["raw"] / f"escenario_{ESCENARIO_ID}.npz"))

tau       = ESC["grilla"]
T0        = int(EVAL_107["T0"])
N_LAGS    = int(EVAL_107["n_lags"])
SEED_BASE = int(HP_107["seed_base"])
assert EVAL_107["objetivo_evaluacion"] == "curva_suavizada", EVAL_107["objetivo_evaluacion"]
assert fpca.M == M_COV, f"los artefactos de {EID_107} tienen M={fpca.M}, no {M_COV}."

print(f"THETA {THETA.shape}   K = {fpca.K}   base {fr_cfg['method']} n_basis={fr_cfg['n_basis']} order={fr_cfg['order']}")
print(f"T0 = {T0}   N_LAGS = {N_LAGS}   seed_base = {SEED_BASE}   M_COV = {M_COV}")
print(f"var. explicada por los {M_COV} scores: {fpca.meta['var_explained']:.2%}")

THETA (1000, 14)   K = 14   base bspline n_basis=14 order=4
T0 = 700   N_LAGS = 2   seed_base = 41232   M_COV = 4
var. explicada por los 4 scores: 91.09%


## §3 Respuesta: coeficientes blanqueados

Se verifica que $\mu+\Phi_o c_t$ reproduzca **exactamente** la curva suavizada de la 107,
`fr.reconstruct(THETA)`.

In [4]:
Phi_o, L_GRAM = mgp.base_ortonormal(fpca.Phi, fpca.W, tau)
theta_bar = THETA[:T0].mean(axis=0)
COEF = (THETA - theta_bar) @ L_GRAM                      # c_t = L'(theta_t - theta_bar)
mu   = fpca.Phi @ theta_bar                              # media en la grilla (train)

X_SUAV = fr.reconstruct(THETA)
err = float(np.abs(mu + COEF @ Phi_o.T - X_SUAV).max())
assert err < 1e-10, f"mu + Phi_o c no reproduce la curva suavizada: {err:.2e}"
print(f"reconstruccion exacta de la curva suavizada: error max {err:.1e}")
print("var(c_k) train:", np.round(COEF[:T0].var(axis=0), 4))

reconstruccion exacta de la curva suavizada: error max 1.8e-15
var(c_k) train: [0.0292 0.0621 0.0975 0.1151 0.1267 0.1144 0.0776 0.1006 0.126  0.1329
 0.1283 0.0704 0.0242 0.0078]


## §4 Diseño y covarianzas proyectadas

In [5]:
if COVARIABLES == "coeficientes":
    Z_FUENTE, NOMBRES_Z = COEF, [f"c{k + 1}" for k in range(COEF.shape[1])]
elif COVARIABLES == "scores":
    Z_FUENTE, NOMBRES_Z = fpca.SCORES[:, :M_COV], [f"xi{j + 1}" for j in range(M_COV)]
else:
    raise ValueError(f"COVARIABLES={COVARIABLES!r}")
DIS = mgp.disenar(COEF, Z_FUENTE, N_LAGS, T0, nombres_z=NOMBRES_Z)
TR = DIS.train
assert DIS.X.shape[1] - 1 == Z_FUENTE.shape[1] * N_LAGS
S_EIG = mgp.covarianzas_proyectadas(Phi_o, tau, CFG_MEZCLA.ells, CFG_MEZCLA.nucleo)

print(f"Y {DIS.Y.shape}   X {DIS.X.shape}   filas train {TR.sum()}   T0 en filas {DIS.T0_ev}")
print("covariables:", ", ".join(DIS.nombres))

# cuanto de la varianza de S(ell) cae en los primeros coeficientes: ell chico = rugoso
tabla_S = pd.DataFrame({f"ell={ell}": np.sort(lam)[::-1][:6] / lam.sum()
                        for ell, (lam, _) in zip(CFG_MEZCLA.ells, S_EIG)}).T
tabla_S.columns = [f"lam{i + 1}" for i in range(tabla_S.shape[1])]
tabla_S.round(3)

Y (998, 14)   X (998, 29)   filas train 698   T0 en filas 698


covariables: c1_lag1, c2_lag1, c3_lag1, c4_lag1, c5_lag1, c6_lag1, c7_lag1, c8_lag1, c9_lag1, c10_lag1, c11_lag1, c12_lag1, c13_lag1, c14_lag1, c1_lag2, c2_lag2, c3_lag2, c4_lag2, c5_lag2, c6_lag2, c7_lag2, c8_lag2, c9_lag2, c10_lag2, c11_lag2, c12_lag2, c13_lag2, c14_lag2


,lam1,lam2,lam3,lam4,lam5,lam6
ell=0.03,0.102,0.101,0.098,0.094,0.090,0.085
ell=0.05,0.134,0.129,0.120,0.110,0.098,0.086
ell=0.08,0.190,0.172,0.148,0.121,0.096,0.074
ell=0.12,0.265,0.219,0.164,0.115,0.078,0.053
ell=0.18,0.368,0.258,0.157,0.090,0.051,0.029
ell=0.25,0.469,0.271,0.131,0.062,0.030,0.016
ell=0.35,0.581,0.256,0.095,0.036,0.015,0.007
ell=0.5,0.697,0.214,0.058,0.018,0.007,0.003


### §4.1 Referencia rápida: lo que explica un lineal con el mismo diseño

Sólo para dimensionar. La comparación formal está en el `exp107_04`.

In [6]:
B_ols, *_ = np.linalg.lstsq(DIS.X[TR], DIS.Y[TR], rcond=None)
res = DIS.Y - DIS.X @ B_ols
r2 = lambda m: 1 - (res[m] ** 2).sum() / ((DIS.Y[m] - DIS.Y[TR].mean(0)) ** 2).sum()
print(f"R^2 L2 del lineal MCO   train {r2(TR):.3f}   test {r2(~TR):.3f}")
d = ESC["diagnostico"]
print(f"generador (107): R^2 L2 oraculo {d['r2_oraculo']:.3f}   lineal {d['r2_lineal']:.3f}")

R^2 L2 del lineal MCO   train 0.423   test 0.405
generador (107): R^2 L2 oraculo 0.446   lineal 0.423


## §5 Contrato para MATLAB

In [7]:
HP = mgp.guardar_contrato(
    PATHS, DIS, S_EIG, Phi_o, mu, CFG_MEZCLA, n_cadenas=N_CHAINS, seed_base=SEED_BASE,
    extra={"fuente": EID_107, "respuesta": "coeficientes_bspline_blanqueados",
           "covariables": COVARIABLES,
           "covariables_fuente": (f"coeficientes c_1..c_{COEF.shape[1]} de {EID_107}" if COVARIABLES == "coeficientes"
                                  else f"scores FPCA 1..{M_COV} de {EID_107}")})

EVAL = dict(EVAL_107)
EVAL.update({"fuente": EID_107, "modelo": "mezcla_gp_funcional",
             "respuesta": "coeficientes_bspline_blanqueados", "m_cov": M_COV})
guardar_config_evaluacion(PATHS, EVAL)

print(f"dims: {HP['dims']}")
print(f"SIG en [{HP['modelo']['SIG'][0]:.4f}, {HP['modelo']['SIG'][-1]:.3f}]   "
      f"NUG en [{HP['modelo']['NUG'][0]:.4f}, {HP['modelo']['NUG'][-1]:.3f}]")
print(f"\n{'=' * 70}\nListo. Siguiente paso, en MATLAB desde esta carpeta:\n"
      f"  >> mezcla_gp_iteracion\ncon EXPERIMENT_ID = {EXPERIMENT_ID}\n{'=' * 70}")

dims: {'n_train': 698, 'K': 14, 'q': 29, 'p': 28, 'nE': 8, 'n_total': 998}
SIG en [0.0220, 2.204]   NUG en [0.0022, 0.055]

Listo. Siguiente paso, en MATLAB desde esta carpeta:
  >> mezcla_gp_iteracion
con EXPERIMENT_ID = experimento_107_1_r01_m04
